In [ ]:
import os
import math
import numpy as np
import networkx as nx
from scipy.spatial import ConvexHull
from scipy import linalg
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import ot
import porespy as ps
from skimage import draw
import sys
import os
sys.path.append(os.path.abspath('..'))
from source.features3D import (
    read_swc, add_ball, swc_to_binary_volume, compute_fractal_dimension, 
    compute_ot_distance_1d, upper_triangular, average_bending_energy_curve
)

In [ ]:
core_path = '../neuron_data/'
folders = ['Basal_ganglia', 'Cerebellum', 'Hippocampus', 'Main_olfactory_bulb', 'Retina']

label_map = {folder: idx for idx, folder in enumerate(folders)}


# Dictionaries to store data for each feature 
feature_data = {
    'N_edg': {folder: [] for folder in folders},
    'Br_l': {folder: [] for folder in folders},
    'Br_dens': {folder: [] for folder in folders},
    'Betw': {folder: [] for folder in folders},
    'G_diam': {folder: [] for folder in folders},
    'Apl': {folder: [] for folder in folders},
    'Assort': {folder: [] for folder in folders},
    'Spec_ent': {folder: [] for folder in folders},
    'Circ': {folder: [] for folder in folders},
    'q1': {folder: [] for folder in folders},
    'q2': {folder: [] for folder in folders},
    'q3': {folder: [] for folder in folders},
    'q4': {folder: [] for folder in folders},
    'Dir_std': {folder: [] for folder in folders},
    'Dir_ent': {folder: [] for folder in folders},
    'Order': {folder: [] for folder in folders},
    'FD': {folder: [] for folder in folders},
    'Alg_Conn': {folder: [] for folder in folders},
    'Energy': {folder: [] for folder in folders}
}
id_data = {folder: [] for folder in folders}


interval_labels = [r'[0, \arccos(3/4)]', r'(\arccos(3/4), \arccos(1/2]]', r'(\arccos(1/2), \arccos(1/4]]', r'(\arccos(1/4), \pi/2]']

# Dictionary to store results
results = {folder: {} for folder in folders}

# Read the SWC file and remove axon (TypeID == 2) for Retina, optionally remove soma (TypeID == 1)
# Helper function to add a ball to the 3D volume
# Create a 3D binary volume from the SWC data
# Function to compute fractal dimension using box-counting
# Function to compute OT distance
# Function to get upper triangular matrix
# Process each neuron
for folder in folders:
    folder_path = os.path.join(core_path, folder, 'CNG version')
    if not os.path.exists(folder_path):
        continue
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".swc"):
            swc_file = os.path.join(folder_path, file_name)
            points = {}
            nodes = {}
            parents = {}
            children = {}
            try:
                # Read SWC file and filter node types
                neuron_name = os.path.basename(swc_file).split('.')[0]
                valid_types = [3] if folder == 'Cerebellum' else [1, 3]
                with open(swc_file, 'r') as f:
                    for line in f:
                        if line.startswith('#') or not line.strip():
                            continue
                        parts = line.split()
                        if len(parts) == 7:
                            index, type_, x, y, z, radius, parent = map(float, parts)
                            type_ = int(type_)
                            if type_ in valid_types or (folder == 'Retina' and type_ != 2):
                                points[index] = (index, type_, x, y, z, radius, parent)
                                nodes[index] = np.array([x, y, z])
                                parents[index] = parent
                                if parent != -1 and parent in nodes:
                                    if parent not in children:
                                        children[parent] = []
                                    children[parent].append(index)
                # Identify branch points, terminals, roots, and soma nodes
                branch_points = [idx for idx in nodes if idx in children and len(children[idx]) > 1]
                terminals = [idx for idx in nodes if idx not in children]
                roots = [idx for idx in nodes if parents[idx] == -1]
                soma_nodes = [idx for idx in nodes if points[idx][1] == 1]
                # Compute branch lengths
                branch_lengths = []
                visited_segments = set()
# Compile features into a DataFrame and save to CSV
data = []
for folder in folders:
    num_files = len(id_data[folder])
    for i in range(num_files):
        row = {
            'id': id_data[folder][i],
            'label': label_map[folder],
        }
        for feature in feature_data:
            row[feature] = feature_data[feature][folder][i]
        data.append(row)
df = pd.DataFrame(data)
df.to_csv('neuron_features.csv', index=False)



# Plot histograms for all features
              
# f, ax = plt.subplots(18, len(folders), figsize=(20, 72))
# colors = ['brown', 'purple', 'green', 'teal', 'orange', 'magenta', 'cyan', 'red', 'gold', 'darkgreen', 'blue', 'coral', 'limegreen', 'darkblue', 'pink', 'gray', 'violet', 'indigo']
# feature_titles = {
# 'number_of_egdes': 'Number of Branches',
# 'avg_branch_length': 'Average Branch Length',
# 'branch_density': 'Edge Density',
# 'mean_betweenness_centrality': 'Mean Betweenness Centrality',
# 'graph_diameter': 'Graph graph_diameter',
# 'average_path_length': 'Average Path Length',
# 'assortativity': 'Assortativity',
# 'spectral_entropy': 'Spectral Entropy',
# 'circuity': 'Circuity',
# 'mass_quartile_1': f'Mass Quartile 1: {interval_labels[0]}',
# 'mass_quartile_2': f'Mass Quartile 2: {interval_labels[1]}',
# 'mass_quartile_3': f'Mass Quartile 3: {interval_labels[2]}',
# 'mass_quartile_4': f'Mass Quartile 4: {interval_labels[3]}',
# 'directional_std_dev': 'Directional Standard Deviation',
# 'directional_entropy': 'Weighted Entropy',
# 'orientation_order': 'Orientation Order',
# 'fractal_dimension': 'Fractal Dimension',
# 'algebraic_connectivity': 'Algebraic Connectivity'
# }
# for i, (feature, data_dict) in enumerate(feature_data.items()):
# for j, folder in enumerate(folders):
# data = data_dict[folder]
# ax[i, j].clear()
# if data:
# avg_value = np.mean(data) if data else 0
# sns.histplot(data, kde=True, bins=20, color=colors[i % len(colors)], alpha=0.7, ax=ax[i, j], stat='probability')
# ax[i, j].axvline(avg_value, color='blue', linestyle='dashed', linewidth=2, label=f'Average: {avg_value:.2f}')
# ax[i, j].set_title(f'{feature_titles[feature]} in {folder}')
# ax[i, j].set_xlabel(feature_titles[feature])
# ax[i, j].set_ylabel('Frequency/Total' if j == 0 else '')
# ax[i, j].legend()
# ax[i, j].grid(axis='y', linestyle='--', alpha=0.7)
# else:
# ax[i, j].set_title(f'No data for {feature_titles[feature]} in {folder}')
# plt.suptitle('Histograms of Features per Brain Region', y=1.02)
# plt.tight_layout()
# plt.show()
# # Compute and plot OT distances for all features
# ot_results = {}
# for feature, data_dict in feature_data.items():
# ot_results[feature] = {}
# for folder1 in folders:
# ot_results[feature][folder1] = {}
# for folder2 in folders:
# data1 = data_dict[folder1]
# data2 = data_dict[folder2]
# if len(data1) > 0 and len(data2) > 0:
# dist = compute_ot_distance_1d(data1, data2)
# ot_results[feature][folder1][folder2] = dist
# df_upper = upper_triangular(pd.DataFrame(ot_results[feature]))
# df_upper = df_upper / np.max(df_upper.max()) if np.max(df_upper.max()) > 0 else df_upper
# plt.figure(figsize=(6, 4))
# sns.heatmap(df_upper, annot=True, cmap='coolwarm', fmt=".2f", mask=df_upper.isnull())
# plt.title(f"Optimal Transport Distance - {feature_titles[feature]}")
# plt.ylabel("Folder 1")
# plt.xlabel("Folder 2")
# plt.show()